# LangSmith CI/CD Evaluation Demo

This notebook demonstrates a complete CI/CD pipeline that automatically evaluates your LangGraph agent whenever code changes are made.

## Overview

```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  Code Change    │────▶│  GitHub Action   │────▶│  LangSmith      │
│  (graph.py)     │     │  Triggers Eval   │     │  Evaluation     │
└─────────────────┘     └──────────────────┘     └────────┬────────┘
                                                          │
                                                          ▼
                        ┌──────────────────┐     ┌─────────────────┐
                        │  PR Comment      │◀────│  Score Report   │
                        │  ✅ PASS / ❌ FAIL│     │  (threshold 0.7)│
                        └──────────────────┘     └─────────────────┘
```

**What you'll see:**
1. Multi-agent system implementation
2. Database tools for the agents
3. LangSmith evaluation setup
4. GitHub Actions workflow
5. Live demo: trigger an eval by opening a PR

---

# Part 1: The Multi-Agent System

Our agent uses a **supervisor pattern** with two specialized sub-agents:
- **Invoice Agent**: Handles billing and purchase queries
- **Music Agent**: Handles music catalog queries

The supervisor routes incoming questions to the appropriate agent.

In [ ]:
# Display the agent implementation
with open("../app/graph.py", "r") as f:
    print(f.read())

## Agent Tools

The agents use tools that query the **Chinook sample database** (a music store database with customers, invoices, albums, and tracks).

In [ ]:
# Display the tools implementation
with open("../app/tools.py", "r") as f:
    print(f.read())

## Try the Agent

Let's run a quick test to see the agent in action.

In [ ]:
import sys
sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv("../.env")

from app import run_graph

# Test query
result = await run_graph({
    "messages": [{"role": "user", "content": "What albums does AC/DC have?"}]
})

print("Response:", result["output"])

---

# Part 2: The Evaluation System

The evaluation uses:
- **LangSmith Dataset**: Test cases with expected answers
- **LLM-as-Judge**: GPT-4o-mini evaluates semantic correctness
- **Threshold**: 70% score required to pass

In [ ]:
# Display the evaluation test
with open("../tests/test_eval.py", "r") as f:
    print(f.read())

## Report Generation

After evals run, a report script queries LangSmith for results and generates a markdown summary.

In [ ]:
# Display the report script
with open("../scripts/report_eval.py", "r") as f:
    print(f.read())

---

# Part 3: GitHub Actions Workflow

The workflow:
1. **Triggers** on PRs that modify `app/graph.py` or `app/tools.py`
2. **Runs** the evaluation test via pytest
3. **Posts** results as a PR comment

In [ ]:
# Display the GitHub Actions workflow
with open("../.github/workflows/evaluate.yml", "r") as f:
    print(f.read())

---

# Part 4: Trigger an Eval (Live Demo)

Now let's trigger an actual evaluation by:
1. Making a code change to the agent
2. Creating a branch and PR
3. Watching the GitHub Action run and post results

**Prerequisites:**
- `gh` CLI installed and authenticated (`gh auth login`)
- Git configured with push access to the repo

In [ ]:
import subprocess
import uuid
import re
from datetime import datetime

def run(cmd: str) -> str:
    """Run a shell command and return output."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd="..")
    if result.returncode != 0:
        print(f"Error: {result.stderr}")
        raise Exception(result.stderr)
    return result.stdout.strip()

In [ ]:
# Configuration
BRANCH_NAME = f"eval-test-{uuid.uuid4().hex[:8]}"
AGENT_FILE = "app/graph.py"

print(f"Branch name: {BRANCH_NAME}")

In [ ]:
# Step 1: Ensure we're on main and up to date
print(run("git checkout main"))
print(run("git pull origin main"))

In [ ]:
# Step 2: Create a new branch
print(run(f"git checkout -b {BRANCH_NAME}"))

In [ ]:
# Step 3: Update the version string in the agent file
with open(f"../{AGENT_FILE}", "r") as f:
    content = f.read()

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
new_version = f"Version: notebook-triggered @ {timestamp}"

updated_content = re.sub(r'Version:.*', new_version, content)

with open(f"../{AGENT_FILE}", "w") as f:
    f.write(updated_content)

print(f"Updated {AGENT_FILE} with: {new_version}")

In [ ]:
# Step 4: Commit the change
run(f"git add {AGENT_FILE}")
print(run('git commit -m "test: trigger eval from notebook"'))

In [ ]:
# Step 5: Push the branch
print(run(f"git push -u origin {BRANCH_NAME}"))

In [ ]:
# Step 6: Create a PR
pr_title = "test: notebook-triggered eval"
pr_body = f"""## Summary
Automated PR created from Jupyter notebook to trigger eval workflow.

**Timestamp:** {timestamp}
**Branch:** {BRANCH_NAME}
"""

pr_url = run(f'gh pr create --title "{pr_title}" --body "{pr_body}"')
print(f"\n{'='*50}")
print(f"PR Created: {pr_url}")
print(f"{'='*50}")

In [ ]:
# Step 7: Open PR in browser to watch the workflow
print("Opening PR in browser...")
run("gh pr view --web")

---

## Cleanup

After watching the eval complete, run these cells to clean up.

In [ ]:
# Close the PR and delete the branch
print(run("gh pr close --delete-branch"))

In [ ]:
# Switch back to main
print(run("git checkout main"))

---

# Summary

You've seen:

1. **Multi-Agent System** (`app/graph.py`)
   - Supervisor pattern with invoice and music agents
   - LangGraph for orchestration

2. **Database Tools** (`app/tools.py`)
   - SQLAlchemy + Chinook database
   - Invoice and music catalog queries

3. **LangSmith Evaluation** (`tests/test_eval.py`)
   - Dataset-driven testing
   - LLM-as-judge for semantic correctness
   - 70% threshold for passing

4. **CI/CD Pipeline** (`.github/workflows/evaluate.yml`)
   - Triggers on agent code changes
   - Runs evals automatically
   - Posts results as PR comments

This pattern ensures your agent quality is validated on every change!